In [51]:
import altair as alt
import pandas as pd
import json
from urllib.request import urlopen

# Enable wider charts
alt.data_transformers.disable_max_rows()
alt.renderers.enable('default')

RendererRegistry.enable('default')

In [52]:
# Download Great Britain Westminster Parliamentary Constituencies GeoJSON
# Source: https://github.com/martinjc/UK-GeoJSON
# This data contains all GB constituencies which we'll filter to extract Wales only

import requests
import time
from pathlib import Path

gb_wpc_url = 'https://raw.githubusercontent.com/martinjc/UK-GeoJSON/master/json/electoral/gb/wpc.json'
gb_output_path = '../data/wales_wpc.geojson'

# Track download success
downloaded = False

# Check if file already exists
if Path(gb_output_path).exists():
    print(f"✓ GB constituencies file already exists at {gb_output_path}")
    print(f"  Source: {gb_wpc_url}")
    print(f"  To force re-download, delete the file and run this cell again")
    downloaded = True
else:
    print(f"Downloading GB Westminster Parliamentary Constituencies...")
    print(f"Source: {gb_wpc_url}")
    
    # Retry logic
    max_retries = 5
    for attempt in range(max_retries):
        try:
            print(f"  Attempt {attempt + 1}/{max_retries}...")
            response = requests.get(gb_wpc_url, timeout=30)
            response.raise_for_status()
            
            # Create directory if needed
            Path(gb_output_path).parent.mkdir(parents=True, exist_ok=True)
            
            # Save the file
            with open(gb_output_path, 'wb') as f:
                f.write(response.content)
            
            print(f"✓ Successfully downloaded to {gb_output_path}")
            downloaded = True
            break
            
        except requests.exceptions.RequestException as e:
            print(f"  ✗ Attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt
                print(f"    Waiting {wait_time} seconds before retry...")
                time.sleep(wait_time)
            else:
                print(f"\n✗ All {max_retries} attempts failed.")
                print(f"  You may need to download manually from: {gb_wpc_url}")
                print(f"  Save as: {gb_output_path}")

✓ GB constituencies file already exists at ../data/wales_wpc.geojson
  Source: https://raw.githubusercontent.com/martinjc/UK-GeoJSON/master/json/electoral/gb/wpc.json
  To force re-download, delete the file and run this cell again


In [53]:
# Load and filter the GB data to extract Wales only
if Path('../data/wales_wpc.geojson').exists():
    print("Loading GB constituencies data...")
    with open('../data/wales_wpc.geojson', 'r', encoding='utf-8') as f:
        gb_data = json.load(f)
    
    # Welsh constituency codes start with 'W' or have Welsh names
    # Filter based on coordinates (Wales is roughly -5.5 to -2.6 longitude, 51.3 to 53.5 latitude)
    wales_features = []
    for feature in gb_data['features']:
        if feature['geometry']['type'] == 'Polygon':
            coords = feature['geometry']['coordinates'][0]
        elif feature['geometry']['type'] == 'MultiPolygon':
            coords = feature['geometry']['coordinates'][0][0]
        else:
            continue
        
        # Get center of feature
        lons = [c[0] for c in coords]
        lats = [c[1] for c in coords]
        center_lon = sum(lons) / len(lons)
        center_lat = sum(lats) / len(lats)
        
        # Check if center is within Wales bounds
        if -5.5 <= center_lon <= -2.6 and 51.3 <= center_lat <= 53.5:
            wales_features.append(feature)
    
    print(f"Extracted {len(wales_features)} Welsh constituencies from {len(gb_data['features'])} GB constituencies")
    
    # Save filtered Wales data
    wales_only = {
        "type": "FeatureCollection",
        "features": wales_features
    }
    
    with open('../data/wales_only.geojson', 'w', encoding='utf-8') as f:
        json.dump(wales_only, f)
    
    print("Saved Wales-only data to ../data/wales_only.geojson")
    
    # Create base map
    base_map = alt.Chart(alt.Data(values=wales_features)).mark_geoshape(
        fill='#e8e8e8',
        stroke='white',
        strokeWidth=0.5
    )
    
    print("✓ Base map created successfully")
    
else:
    print("Warning: Could not load Wales geographic data")
    base_map = None
    print("✗ Failed to create base map - check data source")
    # Create a simple placeholder
    placeholder_data = pd.DataFrame({
        'lon': [-3.5, -3.5, -4.5, -4.5],
        'lat': [51.5, 53.0, 53.0, 51.5]
    })
    base_map = alt.Chart(placeholder_data).mark_line().encode(
        longitude='lon:Q',
        latitude='lat:Q'
    )

Loading GB constituencies data...
Extracted 67 Welsh constituencies from 632 GB constituencies
Saved Wales-only data to ../data/wales_only.geojson
✓ Base map created successfully


In [54]:
# Load WIMD data
wimd_file = '../data/welsh-index-of-multiple-deprivation-wimd-2025-indicator-data-by-lower-layer-super-output-area-lsoa-and-local-authority-community-safety-domain-v4.csv'
wimd_df = pd.read_csv(wimd_file)

# Define major local authorities (22 principal areas)
major_authorities = [
    'Isle of Anglesey',
    'Gwynedd',
    'Conwy',
    'Denbighshire',
    'Flintshire',
    'Wrexham',
    'Powys',
    'Ceredigion',
    'Pembrokeshire',
    'Carmarthenshire',
    'Swansea',
    'Neath Port Talbot',
    'Bridgend',
    'Vale of Glamorgan',
    'Cardiff',
    'Rhondda Cynon Taf',
    'Merthyr Tydfil',
    'Caerphilly',
    'Blaenau Gwent',
    'Torfaen',
    'Monmouthshire',
    'Newport'
]

# Check the unique area names to understand the structure
print("Sample area names:")
print(wimd_df['Area name'].unique()[:20])
print(f"\nTotal unique areas: {wimd_df['Area name'].nunique()}")
print(f"\nTotal records: {len(wimd_df)}")

# Check indicators
print("\nAvailable indicators:")
print(wimd_df['Indicator'].unique())

Sample area names:
['Amlwch Port' 'Amlwch Rural' 'Beaumaris' 'Bodffordd' 'Bodorgan' 'Braint'
 'Bryngwran' 'Brynteg' 'Cadnant (Isle of Anglesey)' 'Cefni' 'Cwm Cadnant'
 'Cyngar' 'Gwyngyll' 'Tref Cybi' 'Kingsland' 'Llanbadrig' 'Llanbedr-goch'
 'Llaneilian' 'Llanfaethlu' 'Llanfair-yn-Neubwll 1']

Total unique areas: 1938

Total records: 17451

Available indicators:
['Police recorded violence with injury per at-risk population'
 'Police recorded public order offences per at-risk population'
 'Police recorded criminal damage per at-risk population'
 'Police recorded violence without injury per at-risk population'
 'Police recorded stalking and harassment per at-risk population'
 'Police recorded antisocial behaviour per at-risk population'
 'Police recorded theft per at-risk population'
 'Police recorded burglary per residential dwelling'
 'Fire incidents per at-risk population']


In [55]:
# Filter data to major authorities only
# The area names include the authority in parentheses, e.g., "Cadnant (Isle of Anglesey)"
# We need to extract areas that match our major authorities list

# Create a pattern to match major authorities
def extract_authority(area_name):
    """Extract the authority name from area name like 'Cadnant (Isle of Anglesey)'"""
    if '(' in area_name and ')' in area_name:
        return area_name.split('(')[1].split(')')[0]
    return area_name

wimd_df['Authority'] = wimd_df['Area name'].apply(extract_authority)

# Filter to only major authorities
wimd_major = wimd_df[wimd_df['Authority'].isin(major_authorities)].copy()

print(f"Filtered to {len(wimd_major)} records from major authorities")
print(f"Authorities found: {sorted(wimd_major['Authority'].unique())}")

# Calculate average values by authority and indicator
wimd_summary = wimd_major.groupby(['Authority', 'Indicator']).agg({
    'Data values': 'mean'
}).reset_index()

# Remove rows with missing data
wimd_summary = wimd_summary[wimd_summary['Data values'].notna()]

print(f"\nSummary has {len(wimd_summary)} records")
print(f"Indicators: {len(wimd_summary['Indicator'].unique())}")

Filtered to 855 records from major authorities
Authorities found: ['Blaenau Gwent', 'Bridgend', 'Caerphilly', 'Cardiff', 'Carmarthenshire', 'Ceredigion', 'Conwy', 'Denbighshire', 'Flintshire', 'Gwynedd', 'Isle of Anglesey', 'Merthyr Tydfil', 'Monmouthshire', 'Neath Port Talbot', 'Newport', 'Pembrokeshire', 'Powys', 'Rhondda Cynon Taf', 'Swansea', 'Torfaen', 'Vale of Glamorgan', 'Wrexham']

Summary has 170 records
Indicators: 9


In [63]:
# Create approximate coordinates for Welsh local authorities (centroids)
authority_coords = {
    'Isle of Anglesey': (-4.35, 53.27),
    'Gwynedd': (-4.13, 52.92),
    'Conwy': (-3.72, 53.16),
    'Denbighshire': (-3.42, 53.18),
    'Flintshire': (-3.15, 53.22),
    'Wrexham': (-3.00, 53.04),
    'Powys': (-3.35, 52.30),
    'Ceredigion': (-4.05, 52.40),
    'Pembrokeshire': (-4.85, 51.80),
    'Carmarthenshire': (-4.15, 51.90),
    'Swansea': (-3.95, 51.62),
    'Neath Port Talbot': (-3.75, 51.65),
    'Bridgend': (-3.58, 51.53),
    'Vale of Glamorgan': (-3.45, 51.42),
    'Cardiff': (-3.18, 51.48),
    'Rhondda Cynon Taf': (-3.43, 51.65),
    'Merthyr Tydfil': (-3.38, 51.75),
    'Caerphilly': (-3.22, 51.65),
    'Blaenau Gwent': (-3.20, 51.78),
    'Torfaen': (-3.03, 51.70),
    'Monmouthshire': (-2.85, 51.75),
    'Newport': (-2.98, 51.59)
}

# Add coordinates to summary
wimd_summary['longitude'] = wimd_summary['Authority'].map(lambda x: authority_coords.get(x, (None, None))[0])
wimd_summary['latitude'] = wimd_summary['Authority'].map(lambda x: authority_coords.get(x, (None, None))[1])

# Remove any rows without coordinates
wimd_summary = wimd_summary[wimd_summary['longitude'].notna()]

print(f"Added coordinates to {len(wimd_summary)} records")
print(f"\nSample data:")
print(wimd_summary.head(10))

Added coordinates to 198 records

Sample data:
       Authority                                          Indicator  \
0  Blaenau Gwent              Fire incidents per at-risk population   
1  Blaenau Gwent  Police recorded antisocial behaviour per at-ri...   
2  Blaenau Gwent  Police recorded burglary per residential dwelling   
3  Blaenau Gwent  Police recorded criminal damage per at-risk po...   
4  Blaenau Gwent  Police recorded public order offences per at-r...   
5  Blaenau Gwent  Police recorded stalking and harassment per at...   
6  Blaenau Gwent       Police recorded theft per at-risk population   
7  Blaenau Gwent  Police recorded violence with injury per at-ri...   
8  Blaenau Gwent  Police recorded violence without injury per at...   
9       Bridgend              Fire incidents per at-risk population   

   Data values  longitude  latitude  Population_2023  
0     0.710000      -3.20     51.78          67356.0  
1          NaN      -3.20     51.78          67356.0  
2     

In [64]:
# Load population data for 2023
pop_file = '../data/statswales-population.csv'
pop_df = pd.read_csv(pop_file, skiprows=7)  # Skip header rows

# Clean up the dataframe
# The first column contains the authority names in the 4th position
pop_df.columns = ['col1', 'col2', 'col3', 'Authority'] + list(pop_df.columns[4:])

# Keep only rows with authority names (non-empty in Authority column)
pop_df = pop_df[pop_df['Authority'].notna() & (pop_df['Authority'] != '')]

# Get 2023 population
pop_df = pop_df[['Authority', 'Mid-year 2023 ']].copy()
pop_df.columns = ['Authority', 'Population_2023']

# Clean authority names (remove extra spaces)
pop_df['Authority'] = pop_df['Authority'].str.strip()

# Convert population to numeric
pop_df['Population_2023'] = pd.to_numeric(pop_df['Population_2023'], errors='coerce')

# Filter to only the 22 principal areas (exclude the Wales total)
pop_df = pop_df[pop_df['Authority'] != 'Wales']

print(f"Loaded population data for {len(pop_df)} authorities")
print(f"\nPopulation data:")
print(pop_df.sort_values('Population_2023', ascending=False))

Loaded population data for 22 authorities

Population data:
            Authority  Population_2023
15            Cardiff         383536.0
11            Swansea         246742.0
16  Rhondda Cynon Taf         241178.0
10    Carmarthenshire         190083.0
18         Caerphilly         176437.0
22            Newport         163628.0
5          Flintshire         155812.0
13           Bridgend         146743.0
12  Neath Port Talbot         142898.0
6             Wrexham         136149.0
14  Vale of Glamorgan         134733.0
7               Powys         134439.0
9       Pembrokeshire         125006.0
2             Gwynedd         119173.0
3               Conwy         114410.0
4        Denbighshire          97156.0
21      Monmouthshire          94572.0
20            Torfaen          93419.0
8          Ceredigion          73050.0
1    Isle of Anglesey          69291.0
19      Blaenau Gwent          67356.0
17     Merthyr Tydfil          58593.0


In [66]:
# Merge population data with WIMD summary
wimd_summary = wimd_summary.merge(pop_df, on='Authority', how='left')

# Create a complete dataset with all authorities for all indicators
# This ensures authorities without data for certain indicators still appear as black dots
all_authorities = sorted(major_authorities)
all_indicators = sorted(wimd_summary['Indicator'].unique())

# Create a complete grid of all authority-indicator combinations
from itertools import product
complete_grid = pd.DataFrame(list(product(all_authorities, all_indicators)), 
                            columns=['Authority', 'Indicator'])

# Merge with existing data to fill in values where available
wimd_summary_complete = complete_grid.merge(
    wimd_summary[['Authority', 'Indicator', 'Data values', 'longitude', 'latitude', 'Population_2023']], 
    on=['Authority', 'Indicator'], 
    how='left'
)

# Fill in coordinates and population for all authorities
for authority in all_authorities:
    mask = wimd_summary_complete['Authority'] == authority
    if wimd_summary_complete.loc[mask, 'longitude'].isna().any():
        wimd_summary_complete.loc[mask, 'longitude'] = authority_coords[authority][0]
        wimd_summary_complete.loc[mask, 'latitude'] = authority_coords[authority][1]
    if wimd_summary_complete.loc[mask, 'Population_2023'].isna().any():
        pop_value = pop_df[pop_df['Authority'] == authority]['Population_2023'].values
        if len(pop_value) > 0:
            wimd_summary_complete.loc[mask, 'Population_2023'] = pop_value[0]

# Replace the summary with the complete version
wimd_summary = wimd_summary_complete

print(f"Complete dataset created with {len(wimd_summary)} records")
print(f"Authorities: {len(wimd_summary['Authority'].unique())}")
print(f"Indicators: {len(wimd_summary['Indicator'].unique())}")
print(f"\nRecords with data: {wimd_summary['Data values'].notna().sum()}")
print(f"Records without data: {wimd_summary['Data values'].isna().sum()}")

Complete dataset created with 198 records
Authorities: 22
Indicators: 9

Records with data: 170
Records without data: 28


In [67]:
# Create interactive map with dropdown selector
# Get list of indicators
indicators = sorted(wimd_summary['Indicator'].unique().tolist())

# Create dropdown for indicator selection
dropdown = alt.binding_select(options=indicators, name='Select Indicator: ')
selection = alt.param(
    name='indicator_selection',
    bind=dropdown,
    value=indicators[0]
)

# Create the points layer for areas WITHOUT data (black dots)
points_no_data = alt.Chart(wimd_summary).mark_circle(
    opacity=1.0,
    stroke='black',
    strokeWidth=1.5,
    color='black'
).encode(
    longitude='longitude:Q',
    latitude='latitude:Q',
    size=alt.Size(
        'Population_2023:Q',
        scale=alt.Scale(range=[300, 1000]),
        legend=alt.Legend(title='Population (2023)')
    ),
    tooltip=[
        alt.Tooltip('Authority:N', title='Authority'),
        alt.Tooltip('Indicator:N', title='Indicator'),
        alt.Tooltip('Population_2023:Q', title='Population (2023)', format=',')
    ]
).transform_filter(
    alt.datum.Indicator == selection
).transform_filter(
    alt.FieldValidPredicate(field='Data values', valid=False)
)

# Create the points layer for areas WITH data
# Points are colored from red (high/worse) to green (low/better)
# Points are sized by population
points_with_data = alt.Chart(wimd_summary).mark_circle(
    opacity=1.0,
    stroke='black',
    strokeWidth=1.5
).encode(
    longitude='longitude:Q',
    latitude='latitude:Q',
    size=alt.Size(
        'Population_2023:Q',
        scale=alt.Scale(range=[300, 1000]),
        legend=alt.Legend(title='Population (2023)')
    ),
    color=alt.Color(
        'Data values:Q',
        scale=alt.Scale(scheme='redyellowgreen', reverse=True),
        legend=alt.Legend(title='Rate')
    ),
    tooltip=[
        alt.Tooltip('Authority:N', title='Authority'),
        alt.Tooltip('Indicator:N', title='Indicator'),
        alt.Tooltip('Data values:Q', title='Rate per 100', format='.2f'),
        alt.Tooltip('Population_2023:Q', title='Population (2023)', format=',')
    ]
).transform_filter(
    alt.datum.Indicator == selection
).transform_filter(
    alt.FieldValidPredicate(field='Data values', valid=True)
).add_params(
    selection
)

# Combine both point layers
points = points_no_data + points_with_data

# Combine base map and points
final_map = (base_map + points).properties(
    width=700,
    height=800,
    title='Welsh Index of Multiple Deprivation - Community Safety Indicators'
).project(
    type='mercator'
).configure_view(
    strokeWidth=0
)

# Save the chart to JSON
final_map.save('../graphs/wales_community_safety_map.json')
print("✓ Chart saved to ../graphs/wales_community_safety_map.json")

final_map

✓ Chart saved to ../graphs/wales_community_safety_map.json


alt.LayerChart(...)